
# 01 — Estimativa de idade com DeepFace e UTKFace

## Objetivo

Este notebook realiza a estimativa de idade das imagens da base de dados UTKFace utilizando o framework DeepFace e seus pesos pré-treinados.

O processamento é realizado individualmente para cada imagem, utilizando o detector facial RetinaFace, com alinhamento habilitado e exigência de detecção facial.

Os resultados são registrados em arquivos CSV, preservando a identificação da imagem, a idade de referência extraída do nome do arquivo, a idade estimada, o erro absoluto e o status da execução.

## Organização experimental

A base UTKFace está dividida em três partições (`part1`, `part2` e `part3`), processadas separadamente por meio da mesma configuração experimental.

A partição utilizada é definida por um parâmetro do notebook, permitindo gerar arquivos de resultados independentes para cada lote.


In [ ]:

# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================

!pip install -q deepface opencv-python pandas

# Observação: caso as dependências sejam instaladas no ambiente do Google Colab, não é necessário adicionar o caracter "!"

In [ ]:

# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import os
import cv2
import pandas as pd

from deepface import DeepFace
from google.colab import drive


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PARTICAO = "part1"

In [ ]:

# ============================================================
# MONTAGEM DO GOOGLE DRIVE
# ============================================================

drive.mount('/content/drive')


In [ ]:

# ============================================================
# CONFIGURAÇÃO DO EXPERIMENTO
# ============================================================

# Partição da base UTKFace que será processada.
# Valores permitidos: part1, part2 ou part3.


PARTICOES_VALIDAS = ["part1", "part2", "part3"]

if PARTICAO not in PARTICOES_VALIDAS:
    raise ValueError(
        f"Partição inválida: {PARTICAO}. "
        f"Utilize uma das opções: {PARTICOES_VALIDAS}"
    )


# Caminho da pasta principal da base UTKFace.
# Ajuste para o local em que a base está armazenada no seu Google Drive.

DIRETORIO_UTKFACE = "/content/drive/MyDrive/CAMINHO/UTKFace"


# Diretório correspondente à partição selecionada.

DIRETORIO_IMAGENS = os.path.join(
    DIRETORIO_UTKFACE,
    PARTICAO
)


# Diretório de armazenamento dos resultados.

DIRETORIO_RESULTADOS = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/resultados/deepface"
)

os.makedirs(DIRETORIO_RESULTADOS, exist_ok=True)


# Nome do arquivo CSV de saída.

ARQUIVO_RESULTADOS = os.path.join(
    DIRETORIO_RESULTADOS,
    f"resultados_deepface_utkface_{PARTICAO}.csv"
)


# Verificação do diretório de entrada.

if not os.path.isdir(DIRETORIO_IMAGENS):
    raise FileNotFoundError(
        f"Diretório da partição não encontrado: {DIRETORIO_IMAGENS}"
    )


print("Configuração do experimento:")
print(f"Partição selecionada: {PARTICAO}")
print(f"Diretório das imagens: {DIRETORIO_IMAGENS}")
print(f"Arquivo de resultados: {ARQUIVO_RESULTADOS}")


In [ ]:
# ATENÇÃO: substituia CAMINHO/UTKFace pelo caminho real da pasta principal da base.

In [ ]:

# ============================================================
# LISTAGEM DAS IMAGENS DA PARTIÇÃO
# ============================================================

imagens = sorted([
    arquivo
    for arquivo in os.listdir(DIRETORIO_IMAGENS)
    if arquivo.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"Partição selecionada: {PARTICAO}")
print(f"Total de imagens encontradas: {len(imagens)}")


In [ ]:

# ============================================================
# CARREGAMENTO DO CHECKPOINT
# ============================================================

INTERVALO_SALVAMENTO = 50

COLUNAS_RESULTADOS = [
    'Arquivo',
    'Idade Real',
    'Idade Estimada',
    'Erro Absoluto',
    'Status'
]

processados_set = set()
resultados = []

if os.path.exists(ARQUIVO_RESULTADOS):

    if os.path.getsize(ARQUIVO_RESULTADOS) == 0:
        raise ValueError(
            "O arquivo de resultados existe, mas está vazio. "
            "Verifique o arquivo antes de continuar."
        )

    df_existente = pd.read_csv(ARQUIVO_RESULTADOS)

    colunas_ausentes = [
        coluna for coluna in COLUNAS_RESULTADOS
        if coluna not in df_existente.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            "O CSV existente não possui todas as colunas esperadas. "
            f"Colunas ausentes: {colunas_ausentes}"
        )

    if df_existente['Arquivo'].isna().any():
        raise ValueError(
            "O CSV existente contém registros sem identificação de arquivo."
        )

    if df_existente['Arquivo'].duplicated().any():
        raise ValueError(
            "O CSV existente contém imagens duplicadas. "
            "Revise o checkpoint antes de continuar."
        )

    resultados = df_existente[COLUNAS_RESULTADOS].to_dict('records')

    processados_set = set(
        df_existente['Arquivo'].astype(str).tolist()
    )

    print("Checkpoint carregado com sucesso.")
    print(f"Imagens já registradas: {len(processados_set)}")

else:
    print("Nenhum checkpoint encontrado.")
    print("O processamento será iniciado do zero.")

print(f"Imagens disponíveis na partição: {len(imagens)}")
print(f"Imagens pendentes: {len(set(imagens) - processados_set)}")


In [ ]:

# ============================================================
# PROCESSAMENTO E INFERÊNCIA COM DEEPFACE
# ============================================================

processados_nesta_execucao = 0

imagens_pendentes = [
    img for img in imagens
    if img not in processados_set
]

print(f"Partição selecionada: {PARTICAO}")
print(f"Total de imagens: {len(imagens)}")
print(f"Imagens já registradas: {len(processados_set)}")
print(f"Imagens pendentes: {len(imagens_pendentes)}")

print("\nIniciando processamento com DeepFace...\n")


for i, img_name in enumerate(imagens_pendentes, 1):

    caminho_imagem = os.path.join(
        DIRETORIO_IMAGENS,
        img_name
    )

    # --------------------------------------------------------
    # EXTRAÇÃO DA IDADE DE REFERÊNCIA
    # --------------------------------------------------------

    try:
        idade_real = int(img_name.split('_')[0])

    except (ValueError, IndexError):

        print(f"[{i}/{len(imagens_pendentes)}] Nome inválido: {img_name}")

        resultados.append({
            'Arquivo': img_name,
            'Idade Real': None,
            'Idade Estimada': None,
            'Erro Absoluto': None,
            'Status': 'Nome inválido'
        })

    else:

        # ----------------------------------------------------
        # LEITURA DA IMAGEM
        # ----------------------------------------------------

        img_array = cv2.imread(caminho_imagem)

        if img_array is None:

            print(f"[{i}/{len(imagens_pendentes)}] Erro ao abrir: {img_name}")

            resultados.append({
                'Arquivo': img_name,
                'Idade Real': idade_real,
                'Idade Estimada': None,
                'Erro Absoluto': None,
                'Status': 'Erro OpenCV'
            })

        else:

            # ------------------------------------------------
            # INFERÊNCIA COM DEEPFACE
            # ------------------------------------------------

            try:

                analise = DeepFace.analyze(
                    img_path=img_array,
                    actions=['age'],
                    detector_backend='retinaface',
                    enforce_detection=True,
                    align=True,
                    silent=True
                )

                if isinstance(analise, list):
                    analise_face = analise[0]
                else:
                    analise_face = analise

                idade_estimada = float(analise_face['age'])

                erro = abs(idade_real - idade_estimada)

                resultados.append({
                    'Arquivo': img_name,
                    'Idade Real': idade_real,
                    'Idade Estimada': round(idade_estimada, 2),
                    'Erro Absoluto': round(erro, 2),
                    'Status': 'Sucesso'
                })

                print(
                    f"[{i}/{len(imagens_pendentes)}] "
                    f"{img_name} | "
                    f"Referência: {idade_real} | "
                    f"Estimada: {idade_estimada:.2f} | "
                    f"Erro: {erro:.2f}"
                )

            except Exception as e:

                print(
                    f"[{i}/{len(imagens_pendentes)}] "
                    f"Erro DeepFace em {img_name}: {str(e)[:150]}"
                )

                resultados.append({
                    'Arquivo': img_name,
                    'Idade Real': idade_real,
                    'Idade Estimada': None,
                    'Erro Absoluto': None,
                    'Status': 'Erro DeepFace'
                })

    # --------------------------------------------------------
    # ATUALIZAÇÃO DO CHECKPOINT
    # --------------------------------------------------------

    processados_set.add(img_name)
    processados_nesta_execucao += 1

    if processados_nesta_execucao % INTERVALO_SALVAMENTO == 0:

        df_checkpoint = pd.DataFrame(
            resultados,
            columns=COLUNAS_RESULTADOS
        )

        df_checkpoint.to_csv(
            ARQUIVO_RESULTADOS,
            index=False
        )

        print(
            f"\n>>> CHECKPOINT SALVO: "
            f"{len(processados_set)} registros identificados.\n"
        )

print("\nEtapa de processamento finalizada.")


In [ ]:

# ============================================================
# SALVAMENTO FINAL E RESUMO DOS RESULTADOS
# ============================================================

df_resultados = pd.DataFrame(
    resultados,
    columns=COLUNAS_RESULTADOS
)

# Salvamento final do CSV
df_resultados.to_csv(
    ARQUIVO_RESULTADOS,
    index=False
)

# ------------------------------------------------------------
# RESUMO DO PROCESSAMENTO
# ------------------------------------------------------------

total_registros = len(df_resultados)

sucessos = df_resultados[
    df_resultados['Status'] == 'Sucesso'
]

total_sucessos = len(sucessos)

total_falhas = total_registros - total_sucessos

print("=" * 55)
print("RESUMO DO EXPERIMENTO — DEEPFACE / UTKFACE")
print("=" * 55)

print(f"Partição processada: {PARTICAO}")
print(f"Total de registros: {total_registros}")
print(f"Inferências bem-sucedidas: {total_sucessos}")
print(f"Registros com falha: {total_falhas}")

# ------------------------------------------------------------
# CÁLCULO DO MAE
# ------------------------------------------------------------

if total_sucessos > 0:

    mae = sucessos['Erro Absoluto'].mean()

    print(f"MAE: {mae:.2f} anos")

else:

    print("MAE: Não calculado (nenhuma inferência bem-sucedida).")

print("=" * 55)
print(f"Arquivo salvo em: {ARQUIVO_RESULTADOS}")
